In [84]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path
import time
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
burgers_dir = Path("html_pages/burger_search")
burgers_dir.mkdir(exist_ok=True)
patties_dir = Path("html_pages/patties_search")
patties_dir.mkdir(exist_ok=True)
headers = {
    "User-Agent": "Mozilla/5.0 (compatible)"
}

# Burgers links

In [ ]:
data_burger = []

for i in tqdm(range(80)):
    url = f'https://www.cooks.com/rec/doc/0,1-{1+i*10},burger,FF.html'
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        filename = urlparse(url).path.strip("/").replace("/", "_") or "index"
        (burgers_dir / f"{filename}").write_text(r.text, encoding="utf-8")

        links = soup.select('a.lnk.b[href^="/recipe"]')

        for a in links:
            href = a.get("href")

            # --- extract ID ---
            parts = href.split("/")

            try:
                recipe_idx = parts.index("recipe")
                recipe_id = parts[recipe_idx + 1]
            except (ValueError, IndexError):
                # malformed href; skip
                continue

            # --- extract name ---
            name = a.get_text(strip=True)

            data_burger.append([recipe_id, name, "www.cooks.com" + href])
        
        time.sleep(np.random.exponential(2.5))
        
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        

with open('data/burger_links.pkl', 'wb') as f:
    pickle.dump(data_burger, f)

# Patties links

In [86]:
data_patties = []

for i in tqdm(range(80)):
    url = f'https://www.cooks.com/rec/doc/0,1-{1+i*10},patties,FF.html'
    
    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        filename = urlparse(url).path.strip("/").replace("/", "_") or "index"
        (patties_dir / f"{filename}").write_text(r.text, encoding="utf-8")

        links = soup.select('a.lnk.b[href^="/recipe"]')

        for a in links:
            href = a.get("href")

            # --- extract ID ---
            parts = href.split("/")

            try:
                recipe_idx = parts.index("recipe")
                recipe_id = parts[recipe_idx + 1]
            except (ValueError, IndexError):
                # malformed href; skip
                continue

            # --- extract name ---
            name = a.get_text(strip=True)

            data_patties.append([recipe_id, name, "www.cooks.com" + href])
        
        time.sleep(np.random.exponential(2.5))
        
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        

with open('data/patties_links.pkl', 'wb') as f:
    pickle.dump(data_patties, f)

100%|██████████| 80/80 [03:38<00:00,  2.74s/it]


In [100]:
with open('data/burger_links.pkl', 'rb') as f:
    data = pickle.load(f)
with open('data/patties_links.pkl', 'rb') as f:
    data_patties = pickle.load(f)

data.extend(data_patties)
data = np.array(data)
np.savetxt('data/combined.csv', data.astype(str), fmt='%s', delimiter=',')
ids, counts = np.unique(data[:,0], return_counts=True)
print(f'Repetitions: {sum(counts>1)}')

Repetitions: 3


# Recipe texts

In [ ]:
# r = requests.get('https://' + data[0][2], timeout=30) # doesn't work anymore :(

In [184]:
urls = data[:,2]
recipes = []

for url in tqdm(urls[:5]):
    try:
        url = "https://web.archive.org/web/20220101000000/https://" + url
        r = requests.get(url, timeout=30)
        soup = BeautifulSoup(r.text, "html.parser")
        html = soup.html
        body = html.body

        centered = body.find("div", class_="centered")
        for div in centered.find_all("div"):
            if div.get("class") == ["row"]:
                for divdiv in div.find_all("div"):
                    if divdiv.get("class") == ["column", "grid_10"]:
                        row = divdiv
                        break


        table = row.find("table", class_="hrecipe")          # first table by default
        tr3 = table.find_all("tr")[2]          # 3rd tr (0-indexed)
        td1 = tr3.find("td")                   # first td
        inner_div = td1.find("div")            # first div inside td

        lines = inner_div.get_text(separator="\n").splitlines()
        recipes.append(lines)
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        recipes.append(['failed to get web page'])

 40%|████      | 2/5 [00:00<00:00,  5.37it/s]


AttributeError: 'NoneType' object has no attribute 'find_all'

In [195]:
np.savetxt('data/combined.csv', data.astype(str), fmt='%s', delimiter=',')

In [198]:
import numpy as np

# Example links array
links = data[:,2]

# Start the HTML content
html_content = "<html>\n<head>\n<title>Links List</title>\n</head>\n<body>\n"
html_content += "<ol>\n"  # ordered list for numbering

# Add each link as a clickable item
for link in links:
    html_content += f'  <li><a href="{'https://' + link}" target="_blank">{'https://' + link}</a></li>\n'

html_content += "</ol>\n</body>\n</html>"

# Save to an HTML file
with open("data/links_list.html", "w") as f:
    f.write(html_content)

print("HTML file 'links_list.html' created successfully!")


HTML file 'links_list.html' created successfully!
